In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

We use Healthcare Spark NLP pretrained models to identify **topics** on a set of about 1000 items extracted from querying **JGH PREM** on the subject of **MSSS**. This notebook contains all the exploration steps. 

# Preliminaries

## Workspace Setup

In [ ]:
!pip install spark-nlp pyspark nltk

In [ ]:
!pip install stylecloud

In [ ]:
# Importing the neccessary libraries
import json
import os

import re
import random

import numpy as np
import pandas as pd

from wordcloud import (
    WordCloud,
    ImageColorGenerator,
    STOPWORDS)

import stylecloud

# Libraries and options for graphs
import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib.colors as mcolors
sns.set_style("darkgrid")

colors = sns.color_palette('PuBuGn')

# Options to display pandas dataframes
pd.options.display.max_colwidth = None
pd.options.display.float_format = '${:,.3f}'.format

In [ ]:
import nltk
nltk.download('stopwords')
from nltk.tokenize import RegexpTokenizer, word_tokenize
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer, WordNetLemmatizer 
from string import punctuation

# Importing Spark libraries and modules

import sparknlp
#import sparknlp_jsl

# Import Spark NLP
import sparknlp
from sparknlp.base import *
from sparknlp.annotator import *
from sparknlp.pretrained import PretrainedPipeline

#from sparknlp_jsl.annotator import *

# Module to display ner results 
# from sparknlp_display import NerVisualizer
# visualiser = NerVisualizer()

from pyspark.sql import SparkSession, Row
from pyspark.sql.types import StructType, StructField, StringType, ArrayType, IntegerType, FloatType, DoubleType, LongType
from pyspark.sql import types as T
from pyspark.sql import functions as F
from pyspark.sql.functions import udf, col, size, regexp_replace, trim, lower, lit, desc, row_number, monotonically_increasing_id, concat_ws, collect_list, concat, explode
from pyspark.sql.window import Window 

from pyspark.ml import Pipeline, PipelineModel
from pyspark.ml.clustering import LDA
from pyspark.ml.linalg import Vectors, SparseVector, DenseVector
from pyspark.ml.feature import Tokenizer, StopWordsRemover, RegexTokenizer, CountVectorizer, IDF

import warnings

In [ ]:
# Settings and parameters for the Spark session
# As included in JSL notebooks

warnings.filterwarnings('ignore')

params = {"spark.driver.memory":"16G", 
          "spark.kryoserializer.buffer.max":"2000M", 
          "spark.driver.maxResultSize":"2000M"} 

print("Spark NLP Version :", sparknlp.version())
#print("Spark NLP_JSL Version :", sparknlp_jsl.version())

# Staring Healthcare Spark NLP session
# spark = sparknlp_jsl.start(license_keys['SECRET'], params=params)
#spark = sparknlp_jsl.start(params=params)
spark = sparknlp.start(params=params)
spark

# Data Import

In [ ]:
! wget -N https://s3.amazonaws.com/auxdata.johnsnowlabs.com/public/resources/en/lemma-corpus-small/lemmas_small.txt -P /tmp

In [ ]:
! wget -N https://s3.amazonaws.com/auxdata.johnsnowlabs.com/public/resources/en/sentiment-corpus/default-sentiment-dict.txt -P /tmp

Here we start our topic modelling. First, we access the data and read it into Spark dataframe.

In [ ]:
# File location and type
file_location = "/kaggle/input/jgh-msss-engonly/sqllab_query_publicmsss_final_mtv_06172025_024124_20251112T192644.csv"
file_type = "csv"

In [ ]:
# CSV options
infer_schema = True
first_row_is_header = True
delimiter = ","

In [ ]:
df = spark.read \
.option("header", first_row_is_header) \
.csv(file_location)

In [ ]:
df = df.withColumn("id", monotonically_increasing_id())

In [ ]:
df.show(10, truncate=10)

Let's checkout what kind of information is stored in our data.

In [ ]:
print(df.columns)

For topic modelling, we need only textual data, thus, we create a new dataframe only with the column of interest.

In [ ]:
import_c = True
text_col = 'answervalue'
print(f"\t\t\t---- Starting the pipeline built for >>> {text_col} <<< with import condition {import_c} ----")

In [ ]:
df = df.withColumn("id", df["id"].cast(IntegerType()))
non_null_index = (df.filter(df[text_col].isNotNull())).select('id')

In [ ]:
df_clean = df.select(text_col).filter(F.col(text_col).isNotNull())
print(f"\n\t1. Cleaning the input for {df.count()} in toal. Null vs. Non-Null = {non_null_index.count()} : {df.count()-non_null_index.count()}. Null = {(df.count()-non_null_index.count()) * 100. / df.count():.2f}%")
print('\t---'*20)

The data that we will use further for the analysis looks as follows:

In [ ]:
df_clean.show(50, truncate=150)

# Data Cleaning

In [ ]:
# Cleaning

@udf
def lower_clean_str(s):
    my_punctuation = '!''\"#$%&\'()*+,-./:;<=>?[\\]^_`{|}~•@â'
    #punc = "!''\"#$&\'()*+,-./:;<=>?@[\\]^_`{|}~"
    lowercased_str = s.lower()
    for c in my_punctuation:
        lowercased_str = lowercased_str.replace(c, '')
    return lowercased_str

In [ ]:
df_clean = df_clean.withColumn('reviewText', lower_clean_str(df_clean.answervalue))

In [ ]:
df_clean.select('reviewText').show(10, truncate=150)

In [ ]:
review_text = df_clean.select('reviewText')
reviews = 'reviewText'

# Spark NLP pipeline

Here we start our NLP pipeline for the task of topic modelling.

## Basic NLP pipeline

Let's start with basic NLP pipeline that clears the data and gets lemmatized unigrams. To understand how you can use Spark NLP annotators (estimators and transformers) for NLP pipeline, you can refer to [Spark NLP documentation](https://nlp.johnsnowlabs.com/) or a corresponding blog post on Topic Modelling with PySpark and Spark NLP.

We will start with [DocumentAssembler](https://sparknlp.org/docs/en/transformers#documentassembler-getting-data-in) that converts data into Spark NLP annotation format that can be used by Spark NLP annotators.

In [ ]:
if import_c:
    from sparknlp.base import DocumentAssembler
    documentAssembler = DocumentAssembler() \
    .setInputCol(reviews) \
    .setOutputCol('document')
    print(f"\n\t2. Attaching DocumentAssembler Transformer to the pipeline")
    print('\t---'*20)

Next step is to tokenize data with [Tokenizer](https://sparknlp.org/docs/en/annotators#tokenizer).

In [ ]:
if import_c:
    from sparknlp.annotator import Tokenizer
    tokenizer = Tokenizer() \
    .setInputCols(['document']) \
    .setOutputCol('tokenized')
    print(f"\n\t3. Attaching Tokenizer Annotator to the pipeline")
    print('\t---'*20)

Further, we clean our data and lowercase it with [Normalizer](https://sparknlp.org/docs/en/annotators#normalizer).

In [ ]:
if import_c:
    from sparknlp.annotator import Normalizer
    normalizer = Normalizer() \
    .setInputCols(['tokenized']) \
    .setOutputCol('normalized') \
    .setLowercase(True)
    print(f"\n\t4. Attaching Normalizer Annotator to the pipeline")
    print('\t---'*20)

We are going to lemmatize our text with pretrained lemming model provided by Spark NLP. 

We can access this model with [LemmatizerModel](https://sparknlp.org/api/python/reference/autosummary/sparknlp/annotator/lemmatizer/index.html).

In [ ]:
if import_c:
    from sparknlp.annotator import LemmatizerModel
    lemmatizer = LemmatizerModel.pretrained() \
    .setInputCols(['normalized']) \
    .setOutputCol('lemmatized')
    print(f"\n\t5. Attaching LemmatizerModel Annotator to the pipeline")
    print('\t---'*20)

Spark NLP doesn't provide stop word list, hence, we will use `nltk` package to download stop words for English.

In [ ]:
if import_c:
    nltk.download("popular")
    my_stopwords = nltk.corpus.stopwords.words('english')
    word_rooter = nltk.stem.snowball.PorterStemmer(ignore_stopwords=False).stem
    print(f"\n\t6. nltk stop-words found")
    print('\t---'*20)

The downloaded list of stop words we will input into [StopWordsCleaner](https://sparknlp.org/docs/en/annotators#stopwordscleaner) that will remove all such words from our lemmatized text.

In [ ]:
if import_c:
    from sparknlp.annotator import StopWordsCleaner
    stopwords_cleaner = StopWordsCleaner() \
    .setInputCols(['lemmatized']) \
    .setOutputCol('unigrams') \
    .setStopWords(my_stopwords)
    print(f"\n\t7. Attaching StopWordsCleaner Annotator to the pipeline")
    print('\t---'*20)

In addition to unigrams, it is good to use n-grams for topic modelling as well since they help to better refine topics. We can get n-grams with [NGramGenerator](https://sparknlp.org/docs/en/annotators#ngramgenerator) in Spark NLP.

In [ ]:
if import_c:
    from sparknlp.annotator import NGramGenerator
    ngrammer = NGramGenerator() \
    .setInputCols(['lemmatized']) \
    .setOutputCol('ngrams') \
    .setN(3) \
    .setEnableCumulative(True) \
    .setDelimiter('_')
    print(f"\n\t8. Attaching NGramGenerator Annotator to the pipeline")
    print('\t---'*20)

We already have our basic NLP pipeline for topic modelling with all necessary steps. However, let's use POS tagger in order to improve our processed data for topic modelling even more with POS tagged data later. For this, we are going to use pretrained POS tagging model provided by Spark NLP. We can access the model with [PerceptronModel](https://sparknlp.org/docs/en/annotators#postagger).

In [ ]:
if import_c:
    from sparknlp.annotator import PerceptronModel
    pos_tagger = PerceptronModel.pretrained('pos_anc') \
    .setInputCols(['document', 'lemmatized']) \
    .setOutputCol('pos')
    print(f"\n\t9. Attaching PerceptronModel Annotator to the pipeline")
    print('\t---'*20)

Now we have everything in Spark NLP annotation format. To be able to process the data further, we need to tranform data with [Finisher](https://sparknlp.org/docs/en/transformers#finisher).

In [ ]:
if import_c:
    from sparknlp.base import Finisher
    finisher = Finisher() \
    .setInputCols(['unigrams', 'ngrams', 'pos']) 
    print(f"\n\t10. Attaching Finisher Transformer to the pipeline")
    print('\t---'*20)

Now we are ready to input everything into a pipeline. **Pipeline** functionality is accessible with PySpark.

In [ ]:
pipeline = Pipeline() \
     .setStages([documentAssembler,                  
                 tokenizer,
                 normalizer,                  
                 lemmatizer,                  
                 stopwords_cleaner, 
                 pos_tagger,
                 ngrammer,  
                 finisher])

First, we will fit all our estimators and then, transform the data with trained models and transformers.

In [ ]:
processed_review = pipeline.fit(review_text).transform(review_text)

Let's look at the data we finally get.

In [ ]:
processed_review.limit(10).show(truncate=40)

## Extended NLP pipeline

Up to now, we have our data in a form of unigrams that are lemmatized, with no stop words in there. I think it is a good idea to incorporate n-grams into our NLP pipeline. We obtained n-grams as one step of our pipeline but now n-grams are messy and have a lot of questionable combinations in there. To tackle this problem, let's filter out strange combinations of words in n-grams based on their POS tags. We can imagine a list of viable combinations like ADJ + NOUN so let's restrict our POS combinations in n-grams to this list. Plus, we can also exclude some POS tags from our unigrams to ensure that we don't use functional words for topic modelling (they can be partially covered by stop words but probably not fully).

Doing this POS-based filtering will significantly reduce the vocabulary size for topic modelling which will speed up the whole processing.

Let's start this processing. First, we need join all our POS tags obtained previously.

In [ ]:
udf_join_arr = F.udf(lambda x: ' '.join(x), T.StringType())
processed_review  = processed_review.withColumn('finished_pos', udf_join_arr(F.col('finished_pos')))

In [ ]:
processed_review.limit(10).show(truncate=40)

Then we start another Spark NLP pipeline in order to get POS tag n-grams that correspond to word n-grams. We start with convertation into Spark NLP annotation format.

In [ ]:
pos_documentAssembler = DocumentAssembler() \
     .setInputCol('finished_pos') \
     .setOutputCol('pos_document')

Then, we tokenize our POS tags.

In [ ]:
pos_tokenizer = Tokenizer() \
     .setInputCols(['pos_document']) \
     .setOutputCol('pos')

And generate n-grams from them in the same way we did that for words. Set N = 3, Trigram. 

In [ ]:
pos_ngrammer = NGramGenerator() \
    .setInputCols(['pos']) \
    .setOutputCol('pos_ngrams') \
    .setN(3) \
    .setEnableCumulative(True) \
    .setDelimiter('_')

Lastly, we are ready to get POS tags ngrams with **Finisher**.

In [ ]:
pos_finisher = Finisher() \
     .setInputCols(['pos', 'pos_ngrams']) 

We create this new Spark NLP pipeline...

In [ ]:
pos_pipeline = Pipeline() \
     .setStages([pos_documentAssembler,                  
                 pos_tokenizer,
                 pos_ngrammer,  
                 pos_finisher])

... and again fit it and transform the data.

In [ ]:
processed_review = pos_pipeline.fit(processed_review).transform(processed_review)

Let's look what kind of data we have to operate with.

In [ ]:
print(processed_review.columns)

And these are our word n-grams with their corresponding pos n-grams.

In [ ]:
processed_review.select('finished_ngrams', 'finished_pos_ngrams').limit(5).show(truncate=80)

Now we are ready to filter out not useful for topic modelling analysis POS tags from our data. Let's create the function that does it for unigrams first. We create the custom Python function and then transform it to PySpark UDF to be used on Spark dataframe.

* https://universaldependencies.org/docs/en/pos/all.html
* https://medium.com/@faisal-fida/the-complete-list-of-pos-tags-in-nltk-with-examples-eb0485f04321

In [ ]:
def filter_pos(words, pos_tags):
    return [word for word, pos in zip(words, pos_tags) 
            if pos in ['JJ', 'NN', 'NNS', 'VB', 'VBP']]

udf_filter_pos = F.udf(filter_pos, T.ArrayType(T.StringType()))

Then, we apply this function on columns with unigrams and their POS tags to get filtered unigrams in a separate dataframe column.

In [ ]:
processed_review = processed_review.withColumn('filtered_unigrams',
                                               udf_filter_pos(F.col('finished_unigrams'), 
                                                              F.col('finished_pos')))

That is how our filtered unigrams look like.

In [ ]:
processed_review.select('filtered_unigrams').limit(5).show(truncate=120)

It is time to filter out improper POS combinations of n-grams. We create the custom function in the same manner as before. Since we deal with bi- and trigrams, we need to restrict tags for both.

In [ ]:
def filter_pos_combs(words, pos_tags):
    return [word for word, pos in zip(words, pos_tags) 
            if (len(pos.split('_')) == 2 and \
                pos.split('_')[0] in ['JJ', 'NN', 'NNS', 'VB', 'VBP'] and \
                 pos.split('_')[1] in ['JJ', 'NN', 'NNS']) \
            or (len(pos.split('_')) == 3 and \
                pos.split('_')[0] in ['JJ', 'NN', 'NNS', 'VB', 'VBP'] and \
                 pos.split('_')[1] in ['JJ', 'NN', 'NNS', 'VB', 'VBP'] and \
                  pos.split('_')[2] in ['NN', 'NNS'])]
    
udf_filter_pos_combs = F.udf(filter_pos_combs, T.ArrayType(T.StringType()))

And we call the function on word and POS n-grams.

In [ ]:
processed_review = processed_review.withColumn('filtered_ngrams',
                                               udf_filter_pos_combs(F.col('finished_ngrams'),
                                                                    F.col('finished_pos_ngrams')))

Below is what we get after filtering for n-grams.

In [ ]:
processed_review.select('filtered_ngrams').limit(5).show(truncate=180)

Now we have unigrams and n-grams stored in different columns in the dataframe. Let's combine them together.

In [ ]:
processed_review = processed_review.withColumn('final', 
                                               concat(F.col('filtered_unigrams'), 
                                                      F.col('filtered_ngrams')))

And this is our final look of the data.

In [ ]:
processed_review.select('final').limit(5).show(truncate=160)

There are additional ways how we could provide cleaner data with Spark NLP for topic modelling analysis. For example:

* You might want to incorporate [SentenceDetector](https://sparknlp.org/docs/en/annotators#sentencedetector) in order to ignore n-grams on the borders of sentences since tokenization in Spark NLP does not account for sentence borders.

* [DependencyParser](https://sparknlp.org/docs/en/annotators#dependency-parsers) also could be used to provide more meaningful n-grams, namely the ones with dependency relation.

* Spell checker also could be incorporated at the early steps of the NLP pipeline for less noisy results. There are various options in Spark NLP such as [NorvigSpellChecker](https://sparknlp.org/docs/en/annotators#norvig-spellchecker) and [SymmetricSpellChecker](https://sparknlp.org/docs/en/annotators#symmetric-spellchecker).

However, in this tutorial we will omit these options since they probably will not bring significant improvements fot topic modelling.

# Vectorization

Now we are set to vectorization of our data. First, we will proceed with TF (term frequency) vectorization with CountVectorizer in PySpark. We fit tf dictionary and then transform the data to vectors of counts.

In [ ]:
tfizer = CountVectorizer(inputCol='final', outputCol='tf_features')
tf_model = tfizer.fit(processed_review)
tf_result = tf_model.transform(processed_review)

In [ ]:
tf_result.show(truncate=15)

After we get TF results, we can account for words that are frequent for all the documents. We can use IDF (inverse document frequency) to lower score of such words.

In [ ]:
idfizer = IDF(inputCol='tf_features', outputCol='tf_idf_features')
idf_model = idfizer.fit(tf_result)
tfidf_result = idf_model.transform(tf_result)

In [ ]:
tfidf_result.show(truncate=12)

# LDA

Finally, we are ready to model topics in our data with [LDA](https://www.geeksforgeeks.org/machine-learning/Latent-Dirichlet-Allocation-and-Topic-Modelling/) (Latent Dirichlet Allocation). To use the algorithm, we have to provide the number of topics we presume our data contains and the number of iterations for the LDA algorithm. Then, we initialize the model and train it.

In [ ]:
num_topics = 20
max_iter = 10

lda = LDA(k=num_topics, maxIter=max_iter, featuresCol='tf_idf_features')
lda_model = lda.fit(tfidf_result)

In [ ]:
lda_topics_df = lda_model.describeTopics(num_topics)

In [ ]:
print('Topics identified by LDA')
lda_topics_df.show(truncate=50)

In [ ]:
vocab = tf_model.vocabulary

def get_words(token_list):
     return [vocab[token_id] for token_id in token_list]
       
udf_to_words = F.udf(get_words, T.ArrayType(T.StringType()))

Let's define the number of top words per topic we would like to see and extract the words with our function.

In [ ]:
num_top_words = 20

topics = lda_model.describeTopics(num_top_words).withColumn('topicWords', udf_to_words(F.col('termIndices')))
topics.select('topic', 'topicWords').show(truncate=150)

And that's it! We are done with topic modelling pipeline on review data.

In [ ]:
topic_scores_df = lda_model.transform(tfidf_result)

In [ ]:
print('Topics scored by LDA')
topic_scores_df.show(truncate=5)

In [ ]:
topic_scores_df.select(['topicDistribution']).show(5, truncate=170)

In [ ]:
topic_scores_df.select(['reviewText']).show(5, truncate=160)

In [ ]:
# determining the topics
def findTopic(v: DenseVector):
    npArr = v.toArray()
    if npArr.sum() == 0: return None
    idxMax = np.argmax(npArr, axis=0)
    return idxMax.item()

In [ ]:
udf_to_topic = F.udf(findTopic, T.IntegerType())
topic_scores_df = topic_scores_df.withColumn('topicNo', udf_to_topic(F.col('topicDistribution')))

In [ ]:
topic_scores_df.show(truncate=5)

In [ ]:
topic_scores_df = topic_scores_df.filter(topic_scores_df.topicNo >= 0)
topic_scores_df.show(truncate=5)

In [ ]:
# Count words in each row, handling nulls and trimming spaces
topic_scores_with_counts_df = topic_scores_df.withColumn(
    "wordCount",
    F.when(
        F.col("reviewText").isNotNull() & (F.length(F.trim(F.col("reviewText"))) > 0),
        F.size(F.split(F.trim(F.col("reviewText")), r"\s+"))
    ).otherwise(0)
)

# Show DataFrame with per-row word counts
topic_scores_with_counts_df.select(['reviewText', 'wordCount']).show(5, truncate=120)

In [ ]:
topic_scores_with_counts_df.show(5, truncate=5)

# Visualization

* Visualization Data Transformation
* prepare data for visualization

In [ ]:
cols = [color for name, color in mcolors.TABLEAU_COLORS.items()] * 4 # more colors: 'mcolors.XKCD_COLORS'

In [ ]:
def count_term_frequency(vectorizedDf, vectorVocabulary): # inputs: tfidf_result, vocab
    '''Count term frequency of Text_FilteredTokens'''
    vocabDf = vectorizedDf.select('finished_unigrams').rdd.flatMap(lambda x: x[0]).toDF(schema=StringType()).toDF('terms')
    vocabFrequency = vocabDf.rdd.countByValue()
    pdf = pd.DataFrame({
                    'term': list(vocabFrequency.keys()),
                    'frequency': list(vocabFrequency.values())
            })
    termFrequencyDf = spark.createDataFrame(pdf).orderBy('frequency', ascending=False)
    def getVocabIndex(term):
        try:
            index = vectorVocabulary.index(term)
            return index
        except:
            return None
    termFrequencyDf = termFrequencyDf.rdd.map(lambda x: Row(term=x['term'][0], frequency=x['frequency'], index=getVocabIndex(x['term'][0]))).toDF()
    return termFrequencyDf

In [ ]:
termFrequencyDf = count_term_frequency(tfidf_result, vocab)
termFrequencyDf.show()

In [ ]:
def get_cloud_df(ldaTopicsDf, vectorVocabulary): # inputs: lda_topics_df, vocab
    '''Prepare df to plot word cloud'''
    # explode array terms of each topic to row
    explodeTermIndices = ldaTopicsDf.select(ldaTopicsDf.topic, explode(ldaTopicsDf.termIndices)).withColumnRenamed("col","termIndices")
    explodeTerm = explodeTermIndices.rdd.map(lambda x: Row(topic=x['topic'], term=vectorVocabulary[x['termIndices']])) #term=Row(vectorVocabulary[x['termIndices']])
    explodeTerm = explodeTerm.toDF().withColumn("id", monotonically_increasing_id())

    explodeTermWeights = ldaTopicsDf.select(explode(ldaTopicsDf.termWeights)).withColumnRenamed("col","termWeights")
    explodeTermWeights = explodeTermWeights.withColumn("id", monotonically_increasing_id())

    cloudDf = explodeTerm.join(explodeTermWeights, 'id', 'outer').drop('id').orderBy('topic')
    return cloudDf

In [ ]:
cloudDf = get_cloud_df(lda_topics_df, vocab)
cloudDf.show()

# Topic Count

In [ ]:
no_topics = 50
topic_count_df = termFrequencyDf.toPandas()
topic_count_df = topic_count_df.drop('index', axis=1)
topic_draw = topic_count_df[:no_topics].sort_values(by="frequency", ascending=True)

In [ ]:
# Set plot style
plt.style.use('seaborn-v0_8')

# Create horizontal bar chart
plt.figure(figsize=(12, 10))
plt.barh(topic_draw['term'], topic_draw['frequency'], color='skyblue')
plt.xlabel('Frequency')
plt.ylabel('Topics')
plt.title('Topic Frequency in Healthcare Context', fontsize=16)
# Add frequency labels to bars
for index, value in enumerate(topic_draw["frequency"]):
    plt.text(value + 5, index, str(value), va="center")
plt.tight_layout()
plt.show()

# Topic Cloud

In [ ]:
def get_cloud_df(ldaTopicsDf, vectorVocabulary): # inputs: lda_topics_df, vocab
    '''Prepare df to plot word cloud'''
    # explode array terms of each topic to row
    explodeTermIndices = ldaTopicsDf.select(ldaTopicsDf.topic, explode(ldaTopicsDf.termIndices)).withColumnRenamed("col","termIndices")
    explodeTerm = explodeTermIndices.rdd.map(lambda x: Row(topic=x['topic'], term=vectorVocabulary[x['termIndices']])) #term=Row(vectorVocabulary[x['termIndices']])
    explodeTerm = explodeTerm.toDF().withColumn("id", monotonically_increasing_id())

    explodeTermWeights = ldaTopicsDf.select(explode(ldaTopicsDf.termWeights)).withColumnRenamed("col","termWeights")
    explodeTermWeights = explodeTermWeights.withColumn("id", monotonically_increasing_id())

    cloudDf = explodeTerm.join(explodeTermWeights, 'id', 'outer').drop('id').orderBy('topic')
    return cloudDf

In [ ]:
topic_cloud_df = get_cloud_df(lda_topics_df, vocab)
topic_cloud_df.show()

In [ ]:
def plot_cloud_by_topic(cloudDf, n_topics):
    '''Plot word cloud of top N words in each topic'''
    cloud_df = cloudDf.toPandas()
    cloud = WordCloud(background_color='white',
                      width=2500,
                      height=1800,
                      max_words=10,
                      colormap='tab10',
                      color_func=lambda *args, **kwargs: cols[i],
                      prefer_horizontal=1.0)

    fig, axes = plt.subplots(4, int(n_topics/4), figsize=(n_topics*0.95, n_topics - 4), sharex=True, sharey=True)
    for i, ax in enumerate(axes.flatten()):
        fig.add_subplot(ax)
        cloud_df_sub = cloud_df.loc[cloud_df.topic == i]
        topic_words = dict(zip(cloud_df_sub.term, cloud_df_sub.termWeights))
        cloud.generate_from_frequencies(topic_words, max_font_size=500)
        plt.gca().imshow(cloud)
        plt.gca().set_title('Topic ' + str(i + 1), fontdict=dict(size=32, color=cols[i]))
        plt.gca().axis('off')
 
    plt.subplots_adjust(wspace=0, hspace=0)
    plt.axis('off')
    plt.margins(x=0, y=0)
    plt.tight_layout()
    plt.show()

In [ ]:
plot_cloud_by_topic(topic_cloud_df, n_topics=20)

# Feedback Length

In [ ]:
import math
def plot_word_count(topicScoresDf, n_topics=20):
    '''Plot word count for each topic'''
    topic_df = topicScoresDf.limit(5000).toPandas()
    review_lens = topic_df.wordCount
    
    plt.figure(figsize=(n_topics*0.95, n_topics - 4))
    # Create histogram
    counts, bins, patches = plt.hist(review_lens, bins=50, color='navy', alpha=0.7)

    # Add value labels above each bar
    for count, bin_left, bin_right in zip(counts[:16], bins[:16], bins[1:16]):
        if count > 0:  # Only label non-empty bins
            plt.text(
                (bin_left + bin_right) / 2,  # X position (center of bin)
                count,                       # Y position (height of bar)
                str(int(count)),             # Label text
                ha='center', va='bottom',    # Center horizontally, place above bar
                fontsize=10, color='black'
        )
        print(f"no. of {count} reviews for <= {math.ceil((bin_left + bin_right) / 2)} word counts in average")    
    
    x_pos = 40
    y_pos = 800
    #plt.yscale("log")
    plt.text(x_pos, y_pos, "Mean   : " + str(round(np.mean(review_lens))), fontsize=16)
    plt.text(x_pos, y_pos-60, "Median : " + str(round(np.median(review_lens))), fontsize=16)
    plt.text(x_pos, y_pos-120, "Stdev   : " + str(round(np.std(review_lens))), fontsize=16)
    plt.text(x_pos, y_pos-180, "1%ile    : " + str(round(np.quantile(review_lens, q=0.01))), fontsize=16)
    plt.text(x_pos, y_pos-240, "99%ile  : " + str(round(np.quantile(review_lens, q=0.99))), fontsize=16)
    plt.gca().set(xlim=(0, 100), ylabel='Number of Reviews', xlabel='Review Word Count')
    plt.title('Distribution of Review Word Counts', fontsize=48)
    plt.tight_layout()
    plt.show()

In [ ]:
plot_word_count(topic_scores_with_counts_df)

# Feedback Length by Topic

In [ ]:
def plot_word_count_by_topic(topicScoresDf, n_topics=20):
    '''plot distribution of word count by LDA topic'''
    topic_df = topicScoresDf.limit(5000).toPandas()
    fig, axes = plt.subplots(4, int(n_topics/4), figsize=(n_topics*0.85, n_topics - 4), sharex=True, sharey=True)
    x_pos, y_pos = 5, 150
    for i, ax in enumerate(axes.flatten()):
        topic_df_sub = topic_df.loc[topic_df.topicNo == i, :]
        review_lens = topic_df_sub.wordCount
        counts, bin_edges, patches = ax.hist(review_lens, bins=25, color=cols[i])
        ax.tick_params(axis='y', labelcolor=cols[i], color=cols[i])
        sns.kdeplot(review_lens, color="black", shade=False, ax=ax.twinx())
        ax.set(xlim=(0, 50), xlabel='Review Word Count')
        ax.set_ylabel('Number of Reviews', color=cols[i])
        #print(f"Topic {str(i + 1)} total word counts = {math.ceil(np.sum(counts * np.diff(bin_edges)))}")
        ax.set_title(f"Topic {str(i + 1)}, total word counts: {math.ceil(np.sum(counts * np.diff(bin_edges)))}", fontdict=dict(size=12, color=cols[i]))

    fig.tight_layout()
    fig.subplots_adjust(top=0.90)
    fig.suptitle('Distribution of Review Word Counts by LDA Topic', fontsize=32)
    plt.show()

In [ ]:
plot_word_count_by_topic(topic_scores_with_counts_df, n_topics=20)